In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

url = ("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/"
"main/cohorts/2026/data/car_fuel_efficiency_2026.csv")

df = pd.read_csv(url)


In [3]:
print(df.isna().sum())

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64


In [4]:
df.dtypes

model_year               int64
origin                     str
fuel_type                  str
drivetrain                 str
num_doors                int64
engine_displacement      int64
num_cylinders            int64
horsepower             float64
vehicle_weight           int64
acceleration           float64
fuel_efficiency_mpg    float64
dtype: object

In [5]:
df = df.fillna(0)

In [6]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [7]:
df_train = df_train.copy()
df_val = df_val.copy()
df_test = df_test.copy()

y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [8]:
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(df_train.to_dict(orient="records"))
X_val = dv.transform(df_val.to_dict(orient="records"))

feature_names = list(dv.get_feature_names_out())
feature_names

['acceleration',
 'drivetrain=All-wheel drive',
 'drivetrain=Front-wheel drive',
 'drivetrain=Rear-wheel drive',
 'engine_displacement',
 'fuel_type=Diesel',
 'fuel_type=Gasoline',
 'fuel_type=Hybrid',
 'horsepower',
 'model_year',
 'num_cylinders',
 'num_doors',
 'origin=Asia',
 'origin=Europe',
 'origin=USA',
 'vehicle_weight']

In [9]:
def rmse(y, predictions):
    return np.sqrt(mean_squared_error(y, predictions))

In [10]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

split_feature = feature_names[dt.tree_.feature[0]]
original_feature = split_feature.split("=", 1)[0]

print("Q1:", original_feature)
print(export_text(dt, feature_names=feature_names))

Q1: model_year
|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [11]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=1)
rf.fit(X_train, y_train)

q2_rmse = rmse(y_val, rf.predict(X_val))
print("Q2:", q2_rmse)

Q2: 1.8332197358745623


In [14]:
estimator_values = [10, 50, 100, 150]
q3_results = []

for n in estimator_values:
    model = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=1)
    model.fit(X_train, y_train)

    score = rmse(y_val, model.predict(X_val))

    q3_results.append({
        "n_estimators": n,
        "rmse": score,
        "rounded_rmse": round(score, 3)
    })

q3_results = pd.DataFrame(q3_results)

print("Q3:")
print(q3_results.to_string(index=False))

best_index = q3_results["rounded_rmse"].idxmin()
print("Best n_estimators:", q3_results.loc[best_index, "n_estimators"])

Q3:
 n_estimators     rmse  rounded_rmse
           10 1.833220         1.833
           50 1.773988         1.774
          100 1.769786         1.770
          150 1.769198         1.769
Best n_estimators: 150


In [15]:
q4_results = []

for depth in [10, 15, 20, 25]:
    scores = []

    for n in estimator_values:
        model = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=1            
        )
        model.fit(X_train, y_train)

        score = rmse(y_val, model.predict(X_val))
        scores.append(score)

    q4_results.append({
        "max_depth": depth,
        "mean_rmse": np.mean(scores)
    })

q4_results = pd.DataFrame(q4_results)

print("Q4:")
print(q4_results.to_string(index=False))

best_index = q4_results["mean_rmse"].idxmin()
print("Best max_depth:", q4_results.loc[best_index, "max_depth"])

Q4:
 max_depth  mean_rmse
        10   1.754392
        15   1.784224
        20   1.786771
        25   1.786305
Best max_depth: 10


In [16]:
model = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
model.fit(X_train, y_train)

importances = pd.Series(model.feature_importances_, index=feature_names).sort_values(ascending=False)

q5_options = ["vehicle_weight", "horsepower", "acceleration", "engine_displacement"]

print("Q5:")
print(importances.loc[q5_options].sort_values(ascending=False))

Q5:
vehicle_weight         0.203313
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
dtype: float64


In [17]:
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_names)

watchlist = [(dtrain, "train"), (dval, "validation")]
q6_results = []

for eta in [0.3, 0.1]:
    xgb_params = {
         "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "nthread": 8,
        "seed": 1,
        "verbosity": 1
    }

    model = xgb.train(params=xgb_params, dtrain=dtrain, num_boost_round=100,
                      evals=watchlist, verbose_eval=False)

    predictions = model.predict(dval)
    score = rmse(y_val, predictions)

    q6_results.append({"eta": eta, "validation_rmse": score})

q6_results = pd.DataFrame(q6_results)

print("Q6:")
print(q6_results.to_string(index=False))

Q6:
 eta  validation_rmse
 0.3         1.826458
 0.1         1.724810
